In [1]:
import pandas as pd

In [2]:
def quantify_event_durations(df, date_col=None, max_gap=1):
    """
    Quantify number and percentage of events by duration,
    allowing gaps of up to `max_gap` days within the same event.

    Parameters
    ----------
    df : pandas.DataFrame
    date_col : str, optional
    max_gap : int
        Maximum number of missing days allowed within an event

    Returns
    -------
    summary : pandas.DataFrame
    event_lengths : pandas.Series
    """

    # Ensure datetime index
    if date_col:
        df = df.copy()
        df[date_col] = pd.to_datetime(df[date_col])
        df = df.sort_values(date_col).set_index(date_col)
    else:
        df = df.sort_index()
        df.index = pd.to_datetime(df.index)

    # Compute difference between consecutive dates
    diff = df.index.to_series().diff().dt.days

    # Allow gaps up to max_gap days
    event_id = (diff > (max_gap + 1)).cumsum()

    # Compute event lengths
    event_lengths = df.groupby(event_id).size()

    # Categorize durations
    one_day = (event_lengths == 1).sum()
    two_days = (event_lengths == 2).sum()
    three_plus = (event_lengths >= 3).sum()

    total_events = len(event_lengths)

    summary = pd.DataFrame({
        "category": ["1 day", "2 days", "≥3 days"],
        "count": [one_day, two_days, three_plus],
        "percentage (%)": [
            100 * one_day / total_events,
            100 * two_days / total_events,
            100 * three_plus / total_events
        ]
    })

    return summary, event_lengths


def get_event_maxima(df, date_col=None, max_gap=1, min_length=3):

    # Preparar índice temporal
    if date_col:
        df = df.copy()
        df[date_col] = pd.to_datetime(df[date_col])
        df = df.sort_values(date_col).set_index(date_col)
    else:
        df = df.sort_index()
        df.index = pd.to_datetime(df.index)

    # Diferencias entre fechas
    diff = df.index.to_series().diff().dt.days

    # Identificar eventos (permitiendo gaps)
    event_id = (diff > (max_gap + 1)).cumsum()

    # Tamaño de cada evento
    event_sizes = df.groupby(event_id).size()

    # Filtrar eventos con longitud mínima
    valid_events = event_sizes[event_sizes >= min_length].index

    # Filtrar dataframe original
    df_filtered = df[event_id.isin(valid_events)]
    event_id_filtered = event_id[event_id.isin(valid_events)]

    # Máximos por evento
    event_max = df_filtered.groupby(event_id_filtered).max()

    # Información adicional del evento
    event_info = df_filtered.groupby(event_id_filtered).apply(
        lambda x: pd.Series({
            "start_date": x.index.min(),
            "end_date": x.index.max(),
            "n_days": len(x),
            "duration_total_days": (x.index.max() - x.index.min()).days + 1
        })
    )

    # Combinar
    result = pd.concat([event_info, event_max], axis=1)

    return result.reset_index(drop=True)

In [3]:
filename  = '/home/cr2/cmtorres/repos/Patagonia-Antarctic_Peninsula/data/EWE_EPE_events_p90.csv'
df_EWE_EPE = pd.read_csv(filename, sep='\t')
df_EWE_EPE['date'] = pd.to_datetime(df_EWE_EPE['date'])
df_EWE_EPE.set_index('date', inplace=True)
df_EWE_EPE

,temperature,p90_har,ETE,precipitation,p95_threshold,EPE
date,,,,,,
1980-02-17,7.854016,5.545154,True,14.086941,8.522828,True
1980-12-23,4.924838,4.753935,True,10.524351,8.522828,True
1981-01-26,8.374053,5.842559,True,10.526930,8.522828,True
1981-12-09,4.326571,4.006913,True,8.930478,8.522828,True
1982-03-02,7.754361,5.539086,True,9.907393,9.876446,True
...,...,...,...,...,...,...
2021-12-09,5.543650,4.343039,True,12.438597,8.522828,True
2021-12-19,5.271071,4.645322,True,8.576263,8.522828,True
2021-12-23,7.964777,4.767314,True,8.661007,8.522828,True


In [4]:
table1 = get_event_maxima(df_EWE_EPE, date_col=None, max_gap=1, min_length=1)
table1



,start_date,end_date,n_days,duration_total_days,temperature,p90_har,ETE,precipitation,p95_threshold,EPE
0,1980-02-17,1980-02-17,1,1,7.854016,5.545154,True,14.086941,8.522828,True
1,1980-12-23,1980-12-23,1,1,4.924838,4.753935,True,10.524351,8.522828,True
2,1981-01-26,1981-01-26,1,1,8.374053,5.842559,True,10.526930,8.522828,True
3,1981-12-09,1981-12-09,1,1,4.326571,4.006913,True,8.930478,8.522828,True
4,1982-03-02,1982-03-03,2,2,7.754361,5.539086,True,21.698622,9.876446,True
...,...,...,...,...,...,...,...,...,...,...
98,2021-12-08,2021-12-09,2,2,6.853923,4.343039,True,14.057010,8.522828,True
99,2021-12-19,2021-12-19,1,1,5.271071,4.645322,True,8.576263,8.522828,True
100,2021-12-23,2021-12-23,1,1,7.964777,4.767314,True,8.661007,8.522828,True
101,2022-02-07,2022-02-07,1,1,7.060025,6.069320,True,11.842184,8.522828,True


In [5]:
from scipy.stats import zscore

# Copia de la tabla de eventos
events = table1.copy()

# Estandarización
events['z_temp'] = zscore(events['temperature'])
events['z_prec'] = zscore(events['precipitation'])

# Índice compuesto
events['compound_index'] = (
    events['z_temp'] +
    events['z_prec']
)

# Ranking
events_rank = (
    events
    .sort_values('compound_index', ascending=False)
    .reset_index(drop=True)
)

events_rank['rank'] = events_rank.index + 1

In [6]:
top10 = events_rank[
    ['rank',
     'start_date',
     'end_date',
     'n_days',
     'temperature',
     'precipitation',
     'compound_index']
].head(10)

display(top10)

,rank,start_date,end_date,n_days,temperature,precipitation,compound_index
0,1,2008-01-25,2008-01-25,1,10.377229,23.679468,4.562696
1,2,2021-02-09,2021-02-11,3,10.445299,23.127605,4.470847
2,3,1988-03-08,1988-03-08,1,6.378127,31.709547,4.137589
3,4,2015-03-24,2015-03-26,2,9.311523,20.376303,3.150394
4,5,1984-12-26,1984-12-27,2,10.067975,17.973000,3.019746
5,6,2019-02-04,2019-02-04,1,12.865652,10.217766,2.806996
6,7,2011-12-16,2011-12-16,1,8.367545,20.919992,2.727437
7,8,1982-03-02,1982-03-03,2,7.754361,21.698622,2.554234
8,9,1990-02-10,1990-02-12,2,8.149093,19.921423,2.361217
9,10,2013-03-03,2013-03-04,2,8.168554,19.437796,2.257191


In [7]:
top10.to_csv('/home/cr2/cmtorres/repos/Patagonia-Antarctic_Peninsula/data/top10_sel.csv', index=None)


In [8]:
top10_recent = top10[
    top10['start_date'] >= '2019-01-01'
]

display(top10_recent)

,rank,start_date,end_date,n_days,temperature,precipitation,compound_index
1,2,2021-02-09,2021-02-11,3,10.445299,23.127605,4.470847
5,6,2019-02-04,2019-02-04,1,12.865652,10.217766,2.806996


In [9]:
events_2020 = events_rank[
    (events_rank['start_date'] >= '2020-01-01') &
    (events_rank['start_date'] < '2021-01-01')
]

display(events_2020[
    ['rank',
     'start_date',
     'end_date',
     'n_days',
     'temperature',
     'precipitation',
     'compound_index']
].sort_values('rank'))

,rank,start_date,end_date,n_days,temperature,precipitation,compound_index
68,69,2020-02-10,2020-02-10,1,6.192674,12.325965,-0.596987
84,85,2020-11-11,2020-11-11,1,4.888386,12.224803,-1.384808
87,88,2020-12-06,2020-12-06,1,5.986446,8.853467,-1.546466
88,89,2020-11-02,2020-11-04,2,4.428102,12.439630,-1.603040


In [10]:
events_2022 = events_rank[
    (events_rank['start_date'] >= '2022-01-01') &
    (events_rank['start_date'] < '2023-01-01')
]

display(events_2022[
    ['rank',
     'start_date',
     'end_date',
     'n_days',
     'temperature',
     'precipitation',
     'compound_index']
].sort_values('rank'))

,rank,start_date,end_date,n_days,temperature,precipitation,compound_index
14,15,2022-02-21,2022-02-21,1,7.792102,18.022362,1.698973
52,53,2022-02-07,2022-02-07,1,7.060025,11.842184,-0.204599
